In [1]:
import numpy as np
import os
from scipy.io import savemat, loadmat
from scipy.stats import qmc
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
from ES_MDA import ES_MDA
import pandas as pd
import utility
import forward_model
import sobol_seq
import copy
from write_script import write_script

In [2]:
def forward(Num_ens):
    data=[]
    for i in range(Num_ens):
        res=pd.read_csv('param'+str(i+1)+'/result.txt',skiprows=8,delimiter ='\s+').iloc[:,:57]
        A=np.hstack(((res.iloc[749,2:]-res.iloc[50,2:]).values*1000,(res.iloc[92,2:]-res.iloc[50,2:]).values*1000,
                    (res.iloc[511,2:]-res.iloc[50,2:]).values*1000))
        data.append(A)
    return np.vstack(data)

In [3]:
# initialization
Num_ens=30
Dim_est=5
Na=4
Alpha=np.array([9.333,7.0,4.0,2.0])
#ncores=10

#load observation data
obs=np.hstack((np.loadtxt('data/SP_a_2.txt')[86+1:142],np.loadtxt('data/SP_b_2.txt')[86+1:142],np.loadtxt('data/SP_c_2.txt')[86+1:142]))

var_obs = np.ones_like(obs)
obs=np.array([obs])
obs=obs.T

# set the error for different kinds of measurements
var_obs[:]=0.5 # SP
var_obs[110:]=1 # SP
R=np.diag(var_obs)

s=np.zeros((Num_ens,Dim_est,Na+1))
data = sobol_seq.i4_sobol_generate(dim_num= Dim_est, n = Num_ens)
# set upper bound for all parameters
para_u=np.array([-12,-4,0.531,-0.523,4])
# set lower bound for all parameters
para_l=np.array([-16,-6,0.342,-2,-1])
array=np.array([[1.00973808e-14, 1.71637457e-06, 2.40416616e+00, 1.65273623e-02, 1.00001211], 
                [6.82053984e-13, 3.81472744e-05, 2.27620469e+00, 1.37049273e-02, 1.00001305e+02],
                [5.13795022e-16, 6.90523086e-06, 2.34279422e+00, 1.08610614e-02, 1.00004874e+03],
                [1.91372524e-14, 4.88817363e-05, 3.35864697e+00, 3.15488773e-02, 1.00001454e+02],
                [9.51927562e-14, 9.91970587e-05, 3.22471446e+00, 2.57672165e-01, 1.03041747e+03],
                [1.81212072e-13, 7.00743827e-06, 2.51376851e+00, 3.62950974e-02, 1.00001883e+01]])
sampler = qmc.Sobol(d=5, scramble=True, seed=8)
raw = sampler.random_base2(m=5)[:24]  # shape (9, 10)
scaled = qmc.scale(raw, l_bounds=para_l, u_bounds=para_u)
data1=np.vstack((np.log10(array),scaled))
s_temp = copy.deepcopy(data1)
s_temp[:, 0] = np.log((data1[:, 0] - para_l[0]) / (para_u[0] - data1[:, 0]))
s_temp[:, 1] = np.log((data1[:, 1] - para_l[1]) / (para_u[1] - data1[:, 1]))
s_temp[:, 2] = np.log((data1[:, 2] - para_l[2]) / (para_u[2] - data1[:, 2]))
s_temp[:, 3] = np.log((data1[:, 3] - para_l[3]) / (para_u[3] - data1[:, 3]))
s_temp[:, 4] = np.log((data1[:, 4] - para_l[4]) / (para_u[4] - data1[:, 4]))
#s_temp=loadmat('../../s_tem3.mat')['s_tem'] #useful for restarting the inversion if it crashed for some reasons, else comment out
s[:,:,0]=s_temp

In [4]:
write=10**(data1.copy())
for i in range (Num_ens):
    write_script(write[i][0],write[i][1],write[i][2],write[i][3],i+1)
    np.savetxt('param'+str(i+1)+'.txt',write[i])

In [ ]:
t =0
sim_obs= forward(Num_ens)# shape of sim_obs (Num_ens,Num_obs)# combine from param1 to param10
np.savetxt('./sim_obs' + str(t) + '.txt', np.mean(sim_obs,axis=0))

rmse=np.sqrt(np.mean((np.mean(sim_obs,axis=0)-obs.flatten())**2))
print('RMSE ite_', t, ' : ', rmse) # not the exact RMSE definition

nse = np.zeros(Num_ens)
for i in range(Num_ens):
    nse[i] = utility.calculate_nse(obs.reshape(-1), sim_obs[i, :].T)
nse_mean = np.mean(nse, 0)
print('NSE ite_', t, ' : ', nse_mean)

# 写入 metrics.txt 文件（追加模式）
with open('metrics.txt', 'a') as f:
    f.write(f'Iteration {t}:\n')
    f.write(f'  RMSE: {rmse:.6f}\n')
    f.write(f'  NSE : {nse_mean:.6f}\n')
    f.write('\n')

s[:,:,t+1] = ES_MDA(Num_ens, s[:,:,t], obs, sim_obs, Alpha[t], R, [], 2)
s_tem=s[:,:,t+1]
savemat('./s_tem' + str(t+1) + '.mat', {'s_tem':s_tem}) # save s for each step
savemat('./s_tem' + str(t) + '.mat', {'s_tem':s[:,:,t]}) # save s for each step
savemat('./sim_obs' + str(t) + '.mat', {'sim_obs':sim_obs}) # save observation for each step

In [ ]:
plt.plot(sim_obs.T,'k',alpha=0.1)
plt.plot(obs,'r')

In [ ]:
s_tempp=copy.deepcopy(s_tem)
s_tempp[:, 0] = para_l[0] + (para_u[0]-para_l[0]) * (np.exp(s_tem[:, 0]) / (1 + np.exp(s_tem[:, 0])))
s_tempp[:, 1] = para_l[1] + (para_u[1]-para_l[1]) * (np.exp(s_tem[:, 1]) / (1 + np.exp(s_tem[:, 1])))
s_tempp[:, 2] = para_l[2] + (para_u[2]-para_l[2]) * (np.exp(s_tem[:, 2]) / (1 + np.exp(s_tem[:, 2])))
s_tempp[:, 3] = para_l[3] + (para_u[3]-para_l[3]) * (np.exp(s_tem[:, 3]) / (1 + np.exp(s_tem[:, 3])))
s_tempp[:, 4] = para_l[4] + (para_u[4]-para_l[4]) * (np.exp(s_tem[:, 4]) / (1 + np.exp(s_tem[:, 4])))
write=10**(s_tempp)

In [ ]:
for i in range (Num_ens):
    write_script(write[i][0],write[i][1],write[i][2],write[i][3],i+1)
    np.savetxt('param'+str(i+1)+'.txt',write[i])

In [ ]:
t =1
sim_obs= forward(Num_ens)# shape of sim_obs (Num_ens,Num_obs)# combine from param1 to param10
np.savetxt('./sim_obs' + str(t) + '.txt', np.mean(sim_obs,axis=0))

rmse=np.sqrt(np.mean((np.mean(sim_obs,axis=0)-obs.flatten())**2))
print('RMSE ite_', t, ' : ', rmse) # not the exact RMSE definition

nse = np.zeros(Num_ens)
for i in range(Num_ens):
    nse[i] = utility.calculate_nse(obs.reshape(-1), sim_obs[i, :].T)
nse_mean = np.mean(nse, 0)
print('NSE ite_', t, ' : ', nse_mean)

# 写入 metrics.txt 文件（追加模式）
with open('metrics.txt', 'a') as f:
    f.write(f'Iteration {t}:\n')
    f.write(f'  RMSE: {rmse:.6f}\n')
    f.write(f'  NSE : {nse_mean:.6f}\n')
    f.write('\n')

s[:,:,t+1] = ES_MDA(Num_ens, s[:,:,t], obs, sim_obs, Alpha[t], R, [], 2)
s_tem=s[:,:,t+1]
savemat('./s_tem' + str(t+1) + '.mat', {'s_tem':s_tem}) # save s for each step
savemat('./sim_obs' + str(t) + '.mat', {'sim_obs':sim_obs}) # save observations for each step

In [ ]:
import matplotlib.pyplot as plt
plt.plot(sim_obs.T,'k',alpha=0.1)
plt.plot(obs,'r')

In [ ]:
s_tempp=copy.deepcopy(s_tem)
s_tempp[:, 0] = para_l[0] + (para_u[0]-para_l[0]) * (np.exp(s_tem[:, 0]) / (1 + np.exp(s_tem[:, 0])))
s_tempp[:, 1] = para_l[1] + (para_u[1]-para_l[1]) * (np.exp(s_tem[:, 1]) / (1 + np.exp(s_tem[:, 1])))
s_tempp[:, 2] = para_l[2] + (para_u[2]-para_l[2]) * (np.exp(s_tem[:, 2]) / (1 + np.exp(s_tem[:, 2])))
s_tempp[:, 3] = para_l[3] + (para_u[3]-para_l[3]) * (np.exp(s_tem[:, 3]) / (1 + np.exp(s_tem[:, 3])))
s_tempp[:, 4] = para_l[4] + (para_u[4]-para_l[4]) * (np.exp(s_tem[:, 4]) / (1 + np.exp(s_tem[:, 4])))
write=10**(s_tempp)

In [ ]:
for i in range (Num_ens):
    write_script(write[i][0],write[i][1],write[i][2],write[i][3],i+1)
    np.savetxt('param'+str(i+1)+'.txt',write[i])

In [ ]:
s[:,:,0]=loadmat('s_tem0.mat')['s_tem']
s[:,:,1]=loadmat('s_tem1.mat')['s_tem']
s[:,:,2]=loadmat('s_tem2.mat')['s_tem']
para_u=np.array([1e-12,1e-4,3.4,0.3,10000])
para_l=np.array([1e-16,1e-6,2.2,0.01,0.1])

In [ ]:
t =2
sim_obs= forward(Num_ens)# shape of sim_obs (Num_ens,Num_obs)# combine from param1 to param10
np.savetxt('./sim_obs' + str(t) + '.txt', np.mean(sim_obs,axis=0))

rmse=np.sqrt(np.mean((np.mean(sim_obs,axis=0)-obs.flatten())**2))
print('RMSE ite_', t, ' : ', rmse) # not the exact RMSE definition

nse = np.zeros(Num_ens)
for i in range(Num_ens):
    nse[i] = utility.calculate_nse(obs.reshape(-1), sim_obs[i, :].T)
nse_mean = np.mean(nse, 0)
print('NSE ite_', t, ' : ', nse_mean)

# 写入 metrics.txt 文件（追加模式）
with open('metrics.txt', 'a') as f:
    f.write(f'Iteration {t}:\n')
    f.write(f'  RMSE: {rmse:.6f}\n')
    f.write(f'  NSE : {nse_mean:.6f}\n')
    f.write('\n')

s[:,:,t+1] = ES_MDA(Num_ens, s[:,:,t], obs, sim_obs, Alpha[t], R, [], 2)
s_tem=s[:,:,t+1]
savemat('./s_tem' + str(t+1) + '.mat', {'s_tem':s_tem}) # save s for each step
savemat('./sim_obs' + str(t) + '.mat', {'sim_obs':sim_obs}) # save observations for each step

In [ ]:
import matplotlib.pyplot as plt
plt.plot(sim_obs.T,'k',alpha=0.1)
plt.plot(obs,'r')

In [ ]:
s_tempp=copy.deepcopy(s_tem)
s_tempp[:, 0] = para_l[0] + (para_u[0]-para_l[0]) * (np.exp(s_tem[:, 0]) / (1 + np.exp(s_tem[:, 0])))
s_tempp[:, 1] = para_l[1] + (para_u[1]-para_l[1]) * (np.exp(s_tem[:, 1]) / (1 + np.exp(s_tem[:, 1])))
s_tempp[:, 2] = para_l[2] + (para_u[2]-para_l[2]) * (np.exp(s_tem[:, 2]) / (1 + np.exp(s_tem[:, 2])))
s_tempp[:, 3] = para_l[3] + (para_u[3]-para_l[3]) * (np.exp(s_tem[:, 3]) / (1 + np.exp(s_tem[:, 3])))
s_tempp[:, 4] = para_l[4] + (para_u[4]-para_l[4]) * (np.exp(s_tem[:, 4]) / (1 + np.exp(s_tem[:, 4])))
write=10**(s_tempp)

In [ ]:
for i in range (Num_ens):
    write_script(write[i][0],write[i][1],write[i][2],write[i][3],i+1)
    np.savetxt('param'+str(i+1)+'.txt',write[i])

In [ ]:
t =3
sim_obs= forward(Num_ens)# shape of sim_obs (Num_ens,Num_obs)# combine from param1 to param10
np.savetxt('./sim_obs' + str(t) + '.txt', np.mean(sim_obs,axis=0))

rmse=np.sqrt(np.mean((np.mean(sim_obs,axis=0)-obs.flatten())**2))
print('RMSE ite_', t, ' : ', rmse) # not the exact RMSE definition

nse = np.zeros(Num_ens)
for i in range(Num_ens):
    nse[i] = utility.calculate_nse(obs.reshape(-1), sim_obs[i, :].T)
nse_mean = np.mean(nse, 0)
print('NSE ite_', t, ' : ', nse_mean)

# 写入 metrics.txt 文件（追加模式）
with open('metrics.txt', 'a') as f:
    f.write(f'Iteration {t}:\n')
    f.write(f'  RMSE: {rmse:.6f}\n')
    f.write(f'  NSE : {nse_mean:.6f}\n')
    f.write('\n')

s[:,:,t+1] = ES_MDA(Num_ens, s[:,:,t], obs, sim_obs, Alpha[t], R, [], 2)
s_tem=s[:,:,t+1]
savemat('./s_tem' + str(t+1) + '.mat', {'s_tem':s_tem}) # save s for each step
savemat('./sim_obs' + str(t) + '.mat', {'sim_obs':sim_obs}) # save observations for each step

In [ ]:
plt.plot(sim_obs.T*1000000,'k',alpha=0.1)
plt.plot(obs,'r')

In [ ]:
s_tempp=copy.deepcopy(s_tem)
s_tempp[:, 0] = para_l[0] + (para_u[0]-para_l[0]) * (np.exp(s_tem[:, 0]) / (1 + np.exp(s_tem[:, 0])))
s_tempp[:, 1] = para_l[1] + (para_u[1]-para_l[1]) * (np.exp(s_tem[:, 1]) / (1 + np.exp(s_tem[:, 1])))
s_tempp[:, 2] = para_l[2] + (para_u[2]-para_l[2]) * (np.exp(s_tem[:, 2]) / (1 + np.exp(s_tem[:, 2])))
s_tempp[:, 3] = para_l[3] + (para_u[3]-para_l[3]) * (np.exp(s_tem[:, 3]) / (1 + np.exp(s_tem[:, 3])))
s_tempp[:, 4] = para_l[4] + (para_u[4]-para_l[4]) * (np.exp(s_tem[:, 4]) / (1 + np.exp(s_tem[:, 4])))
write=10**(s_tempp)

In [ ]:
for i in range (Num_ens):
    write_script(write[i][0],write[i][1],write[i][2],write[i][3],i+1)
    np.savetxt('param'+str(i+1)+'.txt',write[i])

In [ ]:
t =4
sim_obs= forward(Num_ens)# shape of sim_obs (Num_ens,Num_obs)# combine from param1 to param10
np.savetxt('./sim_obs' + str(t) + '.txt', np.mean(sim_obs,axis=0))
savemat('./sim_obs' + str(t) + '.mat', {'sim_obs':sim_obs}) # save observations for each step

In [ ]:
sbatch temp.sh

In [ ]:
git add param{1..30}/data_set_edit.zip

In [ ]:
for /l %i in (1,1,30) do powershell -command "Expand-Archive -Path \"param%i\data_set_edit.zip\" -DestinationPath \"param%i\" -Force; Start-Sleep -Seconds 2" 